In [14]:
import sys

sys.path.append("../src/")

### Samudra

In [15]:
import numpy as np
import torch
import torch.nn as nn

from ocean_emulators.models.base import BaseModel
from ocean_emulators.models.corrector import Corrector
from ocean_emulators.models.modules.blocks import (
    BilinearUpsample,
    CoreBlock,
    TransposedConvUpsample,
)
from ocean_emulators.models.modules.factory import (
    create_block,
    create_downsample,
    create_upsample,
    get_activation_cl,
)
from ocean_emulators.utils.train import pairwise


class Samudra(BaseModel):
    def __init__(self, config, hist, wet):
        super().__init__(
            ch_width=config.ch_width,
            n_out=config.n_out,
            wet=wet,
            hist=hist,
            pred_residuals=config.pred_residuals,
            last_kernel_size=config.last_kernel_size,
            pad=config.pad,
        )

        # Get activation class
        activation = get_activation_cl(config.core_block.activation)

        # Create local copies of config lists that will be reversed
        ch_width = config.ch_width.copy()
        dilation = config.dilation.copy()
        n_layers = config.n_layers.copy()

        # going down
        layers = []
        for i, (a, b) in enumerate(pairwise(ch_width)):
            # Core block
            layers.append(
                create_block(
                    config.core_block.block_type,
                    in_channels=a,
                    out_channels=b,
                    kernel_size=config.core_block.kernel_size,
                    dilation=dilation[i],
                    n_layers=n_layers[i],
                    activation=activation,
                    pad=config.pad,
                    upscale_factor=config.core_block.upscale_factor,
                    norm=config.core_block.norm,
                )
            )
            # Down sampling block
            layers.append(create_downsample(config.down_sampling_block))

        # Middle block
        layers.append(
            create_block(
                config.core_block.block_type,
                in_channels=b,
                out_channels=b,
                kernel_size=config.core_block.kernel_size,
                dilation=dilation[i],
                n_layers=n_layers[i],
                activation=activation,
                pad=config.pad,
                upscale_factor=config.core_block.upscale_factor,
                norm=config.core_block.norm,
            )
        )

        # First upsampling
        layers.append(
            create_upsample(config.up_sampling_block, in_channels=b, out_channels=b)
        )

        # Reverse for upsampling path
        ch_width.reverse()
        dilation.reverse()
        n_layers.reverse()

        # going up
        for i, (a, b) in enumerate(pairwise(ch_width[:-1])):
            layers.append(
                create_block(
                    config.core_block.block_type,
                    in_channels=a,
                    out_channels=b,
                    kernel_size=config.core_block.kernel_size,
                    dilation=dilation[i],
                    n_layers=n_layers[i],
                    activation=activation,
                    pad=config.pad,
                    upscale_factor=config.core_block.upscale_factor,
                    norm=config.core_block.norm,
                )
            )
            layers.append(
                create_upsample(config.up_sampling_block, in_channels=b, out_channels=b)
            )

        # Final conv block
        layers.append(
            create_block(
                config.core_block.block_type,
                in_channels=b,
                out_channels=b,
                kernel_size=config.core_block.kernel_size,
                dilation=dilation[i],
                n_layers=n_layers[i],
                activation=activation,
                pad=config.pad,
                upscale_factor=config.core_block.upscale_factor,
                norm=config.core_block.norm,
            )
        )

        # Final output conv
        layers.append(nn.Conv2d(b, config.n_out, config.last_kernel_size))

        self.layers = nn.ModuleList(layers)
        # self.corrector = Corrector(config.corrector, hist)
        self.num_steps = int(len(config.ch_width) - 1)

    def forward_once(self, fts):
        temp: list[torch.Tensor] = []
        for i in range(self.num_steps):
            temp.append(torch.zeros_like(fts))
        count = 0
        for layer in self.layers:
            crop = fts.shape[2:]
            if isinstance(layer, nn.Conv2d):
                fts = torch.nn.functional.pad(
                    fts, (self.N_pad, self.N_pad, 0, 0), mode=self.pad
                )
                fts = torch.nn.functional.pad(
                    fts, (0, 0, self.N_pad, self.N_pad), mode="constant"
                )
            fts = layer(fts)
            if count < self.num_steps:
                if isinstance(layer, CoreBlock):
                    temp[count] = fts
                    count += 1
            elif count >= self.num_steps:
                if isinstance(layer, BilinearUpsample) or isinstance(
                    layer, TransposedConvUpsample
                ):
                    crop = np.array(fts.shape[2:])
                    shape = np.array(
                        temp[int(2 * self.num_steps - count - 1)].shape[2:]
                    )
                    pads = shape - crop
                    pads = [
                        pads[1] // 2,
                        pads[1] - pads[1] // 2,
                        pads[0] // 2,
                        pads[0] - pads[0] // 2,
                    ]
                    fts = nn.functional.pad(fts, pads)
                    fts += temp[int(2 * self.num_steps - count - 1)]
                    count += 1
        # fts = self.corrector(fts)
        return torch.where(self.wet, fts, 0.0)

In [16]:
from ocean_emulators.config import SamudraConfig

ch_width = [7, 12, 20]
n_out = 6
hist = 1
wet = torch.ones([16, 16])
dilation = [1, 2]
n_layers = [1, 1]
samudra_config = SamudraConfig(ch_width=ch_width, n_out=n_out)
model = Samudra(samudra_config, hist=hist, wet=wet)

In [17]:
from torchinfo import summary

summary(model)

Layer (type:depth-idx)                   Param #
Samudra                                  --
├─ModuleList: 1-1                        --
│    └─ConvNeXtBlock: 2-1                --
│    │    └─Conv2d: 3-1                  96
│    │    └─Sequential: 3-2              9,336
│    └─AvgPool: 2-2                      --
│    │    └─AvgPool2d: 3-3               --
│    └─ConvNeXtBlock: 2-3                --
│    │    └─Conv2d: 3-4                  260
│    │    └─Sequential: 3-5              27,188
│    └─AvgPool: 2-4                      --
│    │    └─AvgPool2d: 3-6               --
│    └─ConvNeXtBlock: 2-5                --
│    │    └─Sequential: 3-7              74,100
│    └─BilinearUpsample: 2-6             --
│    │    └─Upsample: 3-8                --
│    └─ConvNeXtBlock: 2-7                --
│    │    └─Conv2d: 3-9                  252
│    │    └─Sequential: 3-10             73,452
│    └─BilinearUpsample: 2-8             --
│    │    └─Upsample: 3-11               --
│    └─Con

In [18]:
inp = torch.ones([2, 7, 16, 16])
model.forward_once(inp).shape

torch.Size([2, 6, 16, 16])

### CrossFormer

In [19]:
import torch
import logging
import torch.nn.functional as F

from torch import nn, einsum
from einops import rearrange
from einops.layers.torch import Rearrange

#### helpers

In [20]:
# helpers
def cast_tuple(val, length=1):
    return val if isinstance(val, tuple) else ((val,) * length)


def apply_spectral_norm(model):
    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear, nn.ConvTranspose2d)):
            nn.utils.spectral_norm(module)

In [21]:
class UpBlock(nn.Module):
    def __init__(self, in_chans, out_chans, num_groups, num_residuals=2):
        super().__init__()
        self.conv = nn.ConvTranspose2d(in_chans, out_chans, kernel_size=2, stride=2)
        self.output_channels = out_chans

        blk = []
        for i in range(num_residuals):
            blk.append(
                nn.Conv2d(out_chans, out_chans, kernel_size=3, stride=1, padding=1)
            )
            blk.append(nn.GroupNorm(num_groups, out_chans))
            blk.append(nn.SiLU())

        self.b = nn.Sequential(*blk)

    def forward(self, x):
        x = self.conv(x)

        shortcut = x

        x = self.b(x)

        return x + shortcut

In [22]:
# cross embed layer
class CrossEmbedLayer(nn.Module):
    def __init__(self, dim_in, dim_out, kernel_sizes, stride=2):
        super().__init__()
        kernel_sizes = sorted(kernel_sizes)
        num_scales = len(kernel_sizes)

        # calculate the dimension at each scale
        dim_scales = [int(dim_out / (2**i)) for i in range(1, num_scales)]
        dim_scales = [*dim_scales, dim_out - sum(dim_scales)]

        self.convs = nn.ModuleList([])
        for kernel, dim_scale in zip(kernel_sizes, dim_scales):
            self.convs.append(
                nn.Conv2d(
                    dim_in,
                    dim_scale,
                    kernel,
                    stride=stride,
                    padding=(kernel - stride) // 2,
                )
            )

    def forward(self, x):
        fmaps = tuple(map(lambda conv: conv(x), self.convs))
        return torch.cat(fmaps, dim=1)

In [23]:
CrossEmbedLayer(6, 128, [4, 8, 16, 32], 2)

CrossEmbedLayer(
  (convs): ModuleList(
    (0): Conv2d(6, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): Conv2d(6, 32, kernel_size=(8, 8), stride=(2, 2), padding=(3, 3))
    (2): Conv2d(6, 16, kernel_size=(16, 16), stride=(2, 2), padding=(7, 7))
    (3): Conv2d(6, 16, kernel_size=(32, 32), stride=(2, 2), padding=(15, 15))
  )
)

In [24]:
# dynamic positional bias
class DynamicPositionBias(nn.Module):
    def __init__(self, dim):
        super(DynamicPositionBias, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(2, dim),
            nn.LayerNorm(dim),
            nn.ReLU(),
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.ReLU(),
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.ReLU(),
            nn.Linear(dim, 1),
            Rearrange("... () -> ..."),
        )

    def forward(self, x):
        return self.layers(x)


# transformer classes
class LayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.g = nn.Parameter(torch.ones(1, dim, 1, 1))
        self.b = nn.Parameter(torch.zeros(1, dim, 1, 1))

    def forward(self, x):
        var = torch.var(x, dim=1, unbiased=False, keepdim=True)
        mean = torch.mean(x, dim=1, keepdim=True)
        return (x - mean) / (var + self.eps).sqrt() * self.g + self.b


class FeedForward(nn.Module):
    def __init__(self, dim, mult=4, dropout=0.0):
        super(FeedForward, self).__init__()
        self.layers = nn.Sequential(
            LayerNorm(dim),
            nn.Conv2d(dim, dim * mult, 1),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv2d(dim * mult, dim, 1),
        )

    def forward(self, x):
        return self.layers(x)

In [44]:
class Attention(nn.Module):
    def __init__(self, dim, attn_type, window_size, dim_head=32, dropout=0.0):
        super().__init__()
        assert attn_type in {
            "short",
            "long",
        }, "attention type must be one of local or distant"
        heads = dim // dim_head
        self.heads = heads
        self.scale = dim_head**-0.5
        inner_dim = dim_head * heads

        self.attn_type = attn_type
        self.window_size = window_size

        self.norm = LayerNorm(dim)

        self.dropout = nn.Dropout(dropout)

        self.to_qkv = nn.Conv2d(dim, inner_dim * 3, 1, bias=False)
        self.to_out = nn.Conv2d(inner_dim, dim, 1)

        # positions

        self.dpb = DynamicPositionBias(dim // 4)

        # calculate and store indices for retrieving bias

        pos = torch.arange(window_size)
        grid = torch.stack(torch.meshgrid(pos, pos, indexing="ij"))
        grid = rearrange(grid, "c i j -> (i j) c")
        rel_pos = grid[:, None] - grid[None, :]
        rel_pos += window_size - 1
        rel_pos_indices = (rel_pos * torch.tensor([2 * window_size - 1, 1])).sum(dim=-1)

        self.register_buffer("rel_pos_indices", rel_pos_indices, persistent=False)

    def forward(self, x):
        *_, height, width, heads, wsz, device = (
            *x.shape,
            self.heads,
            self.window_size,
            x.device,
        )

        # prenorm

        x = self.norm(x)

        # rearrange for short or long distance attention

        if self.attn_type == "short":
            x = rearrange(x, "b d (h s1) (w s2) -> (b h w) d s1 s2", s1=wsz, s2=wsz)
        elif self.attn_type == "long":
            x = rearrange(x, "b d (l1 h) (l2 w) -> (b h w) d l1 l2", l1=wsz, l2=wsz)

        # queries / keys / values

        q, k, v = self.to_qkv(x).chunk(3, dim=1)

        # split heads

        q, k, v = map(
            lambda t: rearrange(t, "b (h d) x y -> b h (x y) d", h=heads), (q, k, v)
        )
        q = q * self.scale

        sim = einsum("b h i d, b h j d -> b h i j", q, k)

        # add dynamic positional bias

        pos = torch.arange(-wsz, wsz + 1, device=device)
        rel_pos = torch.stack(torch.meshgrid(pos, pos, indexing="ij"))
        rel_pos = rearrange(rel_pos, "c i j -> (i j) c")
        biases = self.dpb(rel_pos.float())
        rel_pos_bias = biases[self.rel_pos_indices]

        sim = sim + rel_pos_bias

        # attend

        attn = sim.softmax(dim=-1)
        attn = self.dropout(attn)

        # merge heads

        out = einsum("b h i j, b h j d -> b h i d", attn, v)
        out = rearrange(out, "b h (x y) d -> b (h d) x y", x=wsz, y=wsz)
        out = self.to_out(out)

        # rearrange back for long or short distance attention

        if self.attn_type == "short":
            out = rearrange(
                out,
                "(b h w) d s1 s2 -> b d (h s1) (w s2)",
                h=height // wsz,
                w=width // wsz,
            )
        elif self.attn_type == "long":
            out = rearrange(
                out,
                "(b h w) d l1 l2 -> b d (l1 h) (l2 w)",
                h=height // wsz,
                w=width // wsz,
            )

        return out

In [60]:
attn_type = "short"
dim = 32
window_size = 4

In [67]:
pos = torch.arange(window_size)
grid = torch.stack(torch.meshgrid(pos, pos, indexing="ij"))
grid = rearrange(grid, "c i j -> (i j) c")
print(grid.shape)
rel_pos = grid[:, None] - grid[None, :]
print(rel_pos.shape)
rel_pos += window_size - 1
rel_pos_indices = (rel_pos * torch.tensor([2 * window_size - 1, 1])).sum(dim=-1)
print(rel_pos_indices)

torch.Size([16, 2])
torch.Size([16, 16, 2])
tensor([[24, 23, 22, 21, 17, 16, 15, 14, 10,  9,  8,  7,  3,  2,  1,  0],
        [25, 24, 23, 22, 18, 17, 16, 15, 11, 10,  9,  8,  4,  3,  2,  1],
        [26, 25, 24, 23, 19, 18, 17, 16, 12, 11, 10,  9,  5,  4,  3,  2],
        [27, 26, 25, 24, 20, 19, 18, 17, 13, 12, 11, 10,  6,  5,  4,  3],
        [31, 30, 29, 28, 24, 23, 22, 21, 17, 16, 15, 14, 10,  9,  8,  7],
        [32, 31, 30, 29, 25, 24, 23, 22, 18, 17, 16, 15, 11, 10,  9,  8],
        [33, 32, 31, 30, 26, 25, 24, 23, 19, 18, 17, 16, 12, 11, 10,  9],
        [34, 33, 32, 31, 27, 26, 25, 24, 20, 19, 18, 17, 13, 12, 11, 10],
        [38, 37, 36, 35, 31, 30, 29, 28, 24, 23, 22, 21, 17, 16, 15, 14],
        [39, 38, 37, 36, 32, 31, 30, 29, 25, 24, 23, 22, 18, 17, 16, 15],
        [40, 39, 38, 37, 33, 32, 31, 30, 26, 25, 24, 23, 19, 18, 17, 16],
        [41, 40, 39, 38, 34, 33, 32, 31, 27, 26, 25, 24, 20, 19, 18, 17],
        [45, 44, 43, 42, 38, 37, 36, 35, 31, 30, 29, 28, 24, 23, 22,

In [71]:
x = torch.arange(16)
x = x.reshape([1, 1, 4, 4])
wsz = 2
print(x)
print("Short")
print(rearrange(x, "b d (h s1) (w s2) -> (b h w) d s1 s2", s1=wsz, s2=wsz))
print("Long")
print(rearrange(x, "b d (h s1) (w s2) -> (b s1 s2) d h w", s1=wsz, s2=wsz))

tensor([[[[ 0,  1,  2,  3],
          [ 4,  5,  6,  7],
          [ 8,  9, 10, 11],
          [12, 13, 14, 15]]]])
Short
tensor([[[[ 0,  1],
          [ 4,  5]]],


        [[[ 2,  3],
          [ 6,  7]]],


        [[[ 8,  9],
          [12, 13]]],


        [[[10, 11],
          [14, 15]]]])
Long
tensor([[[[ 0,  2],
          [ 8, 10]]],


        [[[ 1,  3],
          [ 9, 11]]],


        [[[ 4,  6],
          [12, 14]]],


        [[[ 5,  7],
          [13, 15]]]])


In [72]:
attn = Attention(dim, attn_type, window_size, dim_head=32)
attn(torch.randn([1, dim, 180, 360])).shape

torch.Size([1, 32, 180, 360])

In [25]:
class Transformer(nn.Module):
    def __init__(
        self,
        dim,
        *,
        local_window_size,
        global_window_size,
        depth=4,
        dim_head=32,
        attn_dropout=0.0,
        ff_dropout=0.0,
    ):
        super().__init__()
        self.layers = nn.ModuleList([])

        for _ in range(depth):
            self.layers.append(
                nn.ModuleList(
                    [
                        Attention(
                            dim,
                            attn_type="short",
                            window_size=local_window_size,
                            dim_head=dim_head,
                            dropout=attn_dropout,
                        ),
                        FeedForward(dim, dropout=ff_dropout),
                        Attention(
                            dim,
                            attn_type="long",
                            window_size=global_window_size,
                            dim_head=dim_head,
                            dropout=attn_dropout,
                        ),
                        FeedForward(dim, dropout=ff_dropout),
                    ]
                )
            )

    def forward(self, x):
        for short_attn, short_ff, long_attn, long_ff in self.layers:
            x = short_attn(x) + x
            x = short_ff(x) + x
            x = long_attn(x) + x
            x = long_ff(x) + x

        return x

#### classes

In [26]:
logger = logging.getLogger(__name__)

In [27]:
import torch
import torch.nn.functional as F


class TensorPadding:
    def __init__(self, mode="earth", pad_lat=(40, 40), pad_lon=(40, 40), **kwargs):
        """
        Initialize the TensorPadding class with the specified mode and padding sizes.

        Args:
            mode (str): The padding mode, either 'mirror' or 'earth'.
            pad_lat (list[int]): Padding sizes for the North-South (latitude) dimension [top, bottom].
            pad_lon (list[int]): Padding sizes for the West-East (longitude) dimension [left, right].
        """

        self.mode = mode
        self.pad_NS = pad_lat
        self.pad_WE = pad_lon

    def pad(self, x):
        """
        Apply padding to the tensor based on the specified mode.

        Args:
            x (torch.Tensor): Input tensor of shape (batch, var, time, lat, lon).

        Returns:
            torch.Tensor: The padded tensor.
        """
        if self.mode == "mirror":
            return self._mirror_padding(x)
        elif self.mode == "earth":
            return self._earth_padding(x)

    def unpad(self, x):
        """
        Remove padding from the tensor based on the specified mode.

        Args:
            x (torch.Tensor): Padded tensor of shape (batch, var, time, lat, lon).

        Returns:
            torch.Tensor: The unpadded tensor.
        """
        if self.mode == "mirror":
            return self._mirror_unpad(x)
        elif self.mode == "earth":
            return self._earth_unpad(x)

    def _earth_padding(self, x):
        """
        Apply earth padding to the tensor (poles and circular padding).

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: The padded tensor.
        """
        if any(p > 0 for p in self.pad_NS):
            # 180-degree shift using half the longitude size
            shift_size = int(x.shape[-1] // 2)
            xroll = torch.roll(x, shifts=shift_size, dims=-1)
            # pad poles
            xroll_flip_top = torch.flip(xroll[..., : self.pad_NS[0], :], (-2,))
            xroll_flip_bot = torch.flip(xroll[..., -self.pad_NS[1] :, :], (-2,))
            x = torch.cat([xroll_flip_top, x, xroll_flip_bot], dim=-2)

        if any(p > 0 for p in self.pad_WE):
            x = F.pad(x, (self.pad_WE[0], self.pad_WE[1], 0, 0, 0, 0), mode="circular")

        return x

    def _earth_unpad(self, x):
        """
        Remove earth padding to restore the original tensor size.

        Args:
            x (torch.Tensor): Padded tensor.

        Returns:
            torch.Tensor: The unpadded tensor.
        """
        # unpad along latitude (north-south)
        if any(p > 0 for p in self.pad_NS):
            start_NS = self.pad_NS[0]
            end_NS = -self.pad_NS[1] if self.pad_NS[1] > 0 else None
            x = x[..., start_NS:end_NS, :]

        # unpad along longitude (west-east)
        if any(p > 0 for p in self.pad_WE):
            start_WE = self.pad_WE[0]
            end_WE = -self.pad_WE[1] if self.pad_WE[1] > 0 else None
            x = x[..., :, start_WE:end_WE]

        return x

    def _mirror_padding(self, x):
        """
        Apply mirror padding to the tensor.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: The padded tensor.
        """
        # pad along longitude (west-east)
        if any(p > 0 for p in self.pad_WE):
            pad_lon_left, pad_lon_right = self.pad_WE
            x = F.pad(
                x, pad=(self.pad_WE[0], self.pad_WE[1], 0, 0, 0, 0), mode="circular"
            )

        # pad along latitude (north-south)
        if any(p > 0 for p in self.pad_NS):
            x = F.pad(
                x, pad=(0, 0, self.pad_NS[0], self.pad_NS[1], 0, 0), mode="reflect"
            )

        return x

    def _mirror_unpad(self, x):
        """
        Remove mirror padding to restore the original tensor size.

        Args:
            x (torch.Tensor): Padded tensor.

        Returns:
            torch.Tensor: The unpadded tensor.
        """
        # unpad along latitude (north-south)
        if any(p > 0 for p in self.pad_NS):
            x = x[..., self.pad_NS[0] : -self.pad_NS[1], :]

        # unpad along longitude (west-east)
        if any(p > 0 for p in self.pad_WE):
            x = x[..., :, self.pad_WE[0] : -self.pad_WE[1]]

        return x

In [42]:
# classes
class CrossFormer(BaseModel):
    def __init__(
        self,
        image_height: int = 640,
        patch_height: int = 1,
        image_width: int = 1280,
        patch_width: int = 1,
        pred_residuals=False,
        n_in: int = 157,
        wet=None,
        hist=0,
        n_out: int = 77,
        dim: tuple = (64, 128, 256, 512),
        depth: tuple = (2, 2, 8, 2),
        dim_head: int = 32,
        global_window_size: tuple = (5, 5, 2, 1),
        local_window_size: int = 10,
        cross_embed_kernel_sizes: tuple = ((4, 8, 16, 32), (2, 4), (2, 4), (2, 4)),
        cross_embed_strides: tuple = (4, 2, 2, 2),
        attn_dropout: float = 0.0,
        ff_dropout: float = 0.0,
        use_spectral_norm: bool = True,
        interp: bool = True,
        padding_conf: dict = None,
    ):
        """
        CrossFormer is the base architecture for the WXFormer model. It uses convolutions and long and short distance
        attention layers in the encoder layer and then uses strided transpose convolution blocks for the decoder
        layer.

        Args:
            image_height (int): number of grid cells in the south-north direction.
            patch_height (int): number of grid cells within each patch in the south-north direction.
            image_width (int): number of grid cells in the west-east direction.
            patch_width (int): number of grid cells within each patch in the west-east direction.
            ch_width (list): Channel widths.
            n_out (int): number of outputs.
            dim (tuple): output dimensions of hidden state of each conv/transformer block in the encoder
            depth (tuple): number of attention blocks per encoder layer
            dim_head (int): dimension of each attention head.
            global_window_size (tuple): number of grid cells between cells in long range attention
            local_window_size (tuple): number of grid cells between cells in short range attention
            cross_embed_kernel_sizes (tuple): width of the cross embed kernels in each layer
            cross_embed_strides (tuple): stride of convolutions in each block
            attn_dropout (float): dropout rate for attention layout
            ff_dropout (float): dropout rate for feedforward layers.
            use_spectral_norm (bool): whether to use spectral normalization
            interp (bool): whether to use interpolation
            padding_conf (dict): padding configuration
        """
        super().__init__(
            ch_width=[n_in] + list(dim),
            n_out=n_out,
            wet=wet,
            hist=hist,
            pred_residuals=pred_residuals,
            last_kernel_size=-1,
            pad="",
        )

        dim = tuple(dim)
        depth = tuple(depth)
        global_window_size = tuple(global_window_size)
        cross_embed_kernel_sizes = tuple([tuple(_) for _ in cross_embed_kernel_sizes])
        cross_embed_strides = tuple(cross_embed_strides)

        self.image_height = image_height
        self.image_width = image_width
        self.patch_height = patch_height
        self.patch_width = patch_width
        self.use_spectral_norm = use_spectral_norm
        self.use_interp = interp
        if padding_conf is None:
            padding_conf = {"activate": False}
        self.use_padding = padding_conf["activate"]

        # input channels
        self.input_channels = n_in

        # output channels
        self.output_channels = n_out

        dim = cast_tuple(dim, 4)
        depth = cast_tuple(depth, 4)
        global_window_size = cast_tuple(global_window_size, 4)
        local_window_size = cast_tuple(local_window_size, 4)
        cross_embed_kernel_sizes = cast_tuple(cross_embed_kernel_sizes, 4)
        cross_embed_strides = cast_tuple(cross_embed_strides, 4)

        assert len(dim) == 4
        assert len(depth) == 4
        assert len(global_window_size) == 4
        assert len(local_window_size) == 4
        assert len(cross_embed_kernel_sizes) == 4
        assert len(cross_embed_strides) == 4

        # dimensions
        last_dim = dim[-1]
        first_dim = (
            self.input_channels if (patch_height == 1 and patch_width == 1) else dim[0]
        )
        dims = [first_dim, *dim]
        dim_in_and_out = tuple(zip(dims[:-1], dims[1:]))

        # allocate cross embed layers
        self.layers = nn.ModuleList([])

        # loop through hyperparameters
        for (
            dim_in,
            dim_out,
        ), num_layers, global_wsize, local_wsize, kernel_sizes, stride in zip(
            dim_in_and_out,
            depth,
            global_window_size,
            local_window_size,
            cross_embed_kernel_sizes,
            cross_embed_strides,
        ):
            # create CrossEmbedLayer
            cross_embed_layer = CrossEmbedLayer(
                dim_in=dim_in, dim_out=dim_out, kernel_sizes=kernel_sizes, stride=stride
            )

            # create Transformer
            transformer_layer = Transformer(
                dim=dim_out,
                local_window_size=local_wsize,
                global_window_size=global_wsize,
                depth=num_layers,
                dim_head=dim_head,
                attn_dropout=attn_dropout,
                ff_dropout=ff_dropout,
            )

            # append everything
            self.layers.append(nn.ModuleList([cross_embed_layer, transformer_layer]))

        if self.use_padding:
            self.padding_opt = TensorPadding(**padding_conf)

        # =================================================================================== #

        self.up_block1 = UpBlock(1 * last_dim, last_dim // 2, dim[0])
        self.up_block2 = UpBlock(2 * (last_dim // 2), last_dim // 4, dim[0])
        self.up_block3 = UpBlock(2 * (last_dim // 4), last_dim // 8, dim[0])
        self.up_block4 = nn.ConvTranspose2d(
            2 * (last_dim // 8),
            self.output_channels,
            kernel_size=4,
            stride=2,
            padding=1,
        )

        if self.use_spectral_norm:
            logger.info("Adding spectral norm to all conv and linear layers")
            apply_spectral_norm(self)

    def forward_once(self, x):
        x_copy = None
        # if self.use_post_block:  # copy tensor to feed into postBlock later
        #     x_copy = x.clone().detach()
        print("Input: ", x.shape)

        if self.use_padding:
            x = self.padding_opt.pad(x)

        print("Input post padding: ", x.shape)

        encodings = []
        for cel, transformer in self.layers:
            x = cel(x)
            print("Input post cross embedding: ", x.shape)
            x = transformer(x)
            print("Input post transformer: ", x.shape)
            encodings.append(x)

        x = self.up_block1(x)
        print(x.shape)
        x = torch.cat([x, encodings[2]], dim=1)
        print(x.shape)
        x = self.up_block2(x)
        print(x.shape)
        x = torch.cat([x, encodings[1]], dim=1)
        print(x.shape)
        x = self.up_block3(x)
        print(x.shape)
        x = torch.cat([x, encodings[0]], dim=1)
        print(x.shape)
        x = self.up_block4(x)
        print(x.shape)

        if self.use_padding:
            x = self.padding_opt.unpad(x)
            print(x.shape)

        if self.use_interp:
            x = F.interpolate(
                x, size=(self.image_height, self.image_width), mode="bilinear"
            )
            print("Interpolation: ", x.shape)

        # x = x.unsqueeze(2)
        print(x.shape)

        return x

In [43]:
image_height = 180  # 640, 192
image_width = 360  # 1280, 288
n_in = 6
n_out = 5
hist = 1
wet = torch.ones([n_out, image_height, image_width])

input_tensor = torch.randn(
    1,
    n_in,
    image_height,
    image_width,
)


model = CrossFormer(
    image_height=image_height,
    image_width=image_width,
    n_in=6,
    n_out=5,
    wet=wet,
    hist=hist,
    dim=(128, 256, 512, 1024),
    depth=(2, 2, 18, 2),
    global_window_size=(8, 4, 2, 1),
    local_window_size=4,
    cross_embed_kernel_sizes=((4, 8, 16, 32), (2, 4), (2, 4), (2, 4)),
    cross_embed_strides=(2, 2, 2, 2),
    attn_dropout=0.0,
    ff_dropout=0.0,
    padding_conf={"activate": True, "pad_lat": (6, 6), "pad_lon": (12, 12)},
)

num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters in the model: {num_params}")

y_pred = model.forward_once(input_tensor)
print("Predicted shape:", y_pred.shape)

Number of parameters in the model: 186220085
Input:  torch.Size([1, 6, 180, 360])
Input post padding:  torch.Size([1, 6, 192, 384])
Input post cross embedding:  torch.Size([1, 128, 96, 192])
Input post transformer:  torch.Size([1, 128, 96, 192])
Input post cross embedding:  torch.Size([1, 256, 48, 96])
Input post transformer:  torch.Size([1, 256, 48, 96])
Input post cross embedding:  torch.Size([1, 512, 24, 48])
Input post transformer:  torch.Size([1, 512, 24, 48])
Input post cross embedding:  torch.Size([1, 1024, 12, 24])
Input post transformer:  torch.Size([1, 1024, 12, 24])
torch.Size([1, 512, 24, 48])
torch.Size([1, 1024, 24, 48])
torch.Size([1, 256, 48, 96])
torch.Size([1, 512, 48, 96])
torch.Size([1, 128, 96, 192])
torch.Size([1, 256, 96, 192])
torch.Size([1, 5, 192, 384])
torch.Size([1, 5, 180, 360])
Interpolation:  torch.Size([1, 5, 180, 360])
torch.Size([1, 5, 180, 360])
Predicted shape: torch.Size([1, 5, 180, 360])


In [41]:
from torchinfo import summary

summary(model)

Layer (type:depth-idx)                                            Param #
CrossFormer                                                       --
├─ModuleList: 1-1                                                 --
│    └─ModuleList: 2-1                                            --
│    │    └─CrossEmbedLayer: 3-1                                  141,440
│    │    └─Transformer: 3-2                                      801,284
│    └─ModuleList: 2-2                                            --
│    │    └─CrossEmbedLayer: 3-3                                  327,936
│    │    └─Transformer: 3-4                                      3,191,812
│    └─ModuleList: 2-3                                            --
│    │    └─CrossEmbedLayer: 3-5                                  1,311,232
│    │    └─Transformer: 3-6                                      114,665,508
│    └─ModuleList: 2-4                                            --
│    │    └─CrossEmbedLayer: 3-7                            